In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../")
from ccfm.ccfm import (
    load_cfm_traces, 
    make_3d_fault_mesh, 
    load_nshm_traces, 
    load_nrcan_traces,
    make_tri_mesh,
    write_cfm_tri_meshes,
    write_cfm_trace_geojson,
    convert_cfm_geojson,
)

In [ ]:
nshm_cfm = "../input_fault_data/nshm_pnw_traces_elev.geojson"

In [ ]:
us_flts = load_nshm_traces(nshm_cfm)

In [ ]:
us_flts[0]

In [ ]:
can_faults = "../input_fault_data/WesternCanada_QuaternaryFaults_elev.geojson"

In [ ]:
canada_include_ids = (
    "wc022",
    "wc023",
    "wc024",
)

In [ ]:
can_flts = load_nrcan_traces(can_faults, include_ids=canada_include_ids)

In [ ]:
can_flts[0]

In [ ]:
flts = us_flts + can_flts

len(flts)

In [ ]:
%%time

meshes = []
tri_meshes = []

for i, flt in enumerate(flts):
    if i < len(flts) - 1:
        print(f"working on fault {str(i+1).zfill(3)} / {len(flts)}", end='\r')
    else:
        print(f"working on fault {str(i+1).zfill(3)} / {len(flts)}", end='\n')
    try:
        flt_mesh = make_3d_fault_mesh(flt, pt_distance=5.0)
        tri_mesh = make_tri_mesh(flt_mesh)
        meshes.append(flt_mesh)
        tri_meshes.append(tri_mesh)
    except Exception as e:
        print(e)
        print(flt['properties']['id'])
        print(i)

In [ ]:
write_cfm_tri_meshes("../crescent_cfm_files/crescent_cfm_crustal_3d_5km.geojson", tri_meshes, flts, minify=True)

In [ ]:
for i, fault in enumerate(flts):
    fault['geometry']['coordinates'] = meshes[i][0]

In [ ]:
write_cfm_trace_geojson("../crescent_cfm_files/crescent_cfm_crustal_traces.geojson", flts, minify=False)

In [ ]:
run add_metadata.py

In [ ]:
convert_cfm_geojson("../crescent_cfm_files/crescent_cfm_crustal_traces.geojson")

In [ ]:
convert_cfm_geojson("../crescent_cfm_files/crescent_cfm_crustal_3d.geojson",
                    outfile_types=['geopackage'])